In [1]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
from lifelines.utils import concordance_index
from sksurv.ensemble import RandomSurvivalForest

train_features = pd.read_pickle("../data/processed/train_features.pkl")
test_features = pd.read_pickle("../data/processed/test_features.pkl")
y_train = pd.read_pickle("../data/processed/y_train.pkl")
y_test = pd.read_pickle("../data/processed/y_test.pkl")

print("Data loaded successfully!")
print("Train features shape:", train_features.shape)

Data loaded successfully!
Train features shape: (1522, 680)


In [2]:
print("Features columns preview:", train_features.columns[:10])

clinical_cols = [col for col in train_features.columns if any(term in col.lower() for term in ['npi', 'cohort', 'chemo', 'hormone', 'radio', 'age', 'size', 'grade', 'stage'])]
molecular_cols = [col for col in train_features.columns if col not in clinical_cols]

X_train_clin = train_features[clinical_cols]
X_test_clin = test_features[clinical_cols]

X_train_mol = train_features[molecular_cols] if len(molecular_cols) > 0 else None
X_test_mol = test_features[molecular_cols] if len(molecular_cols) > 0 else None

X_train_comb = train_features
X_test_comb = test_features

print("y_train preview:")
print(y_train)

Features columns preview: Index(['age_at_diagnosis', 'type_of_breast_surgery', 'cancer_type_detailed',
       'cellularity', 'er_status', 'her2_status', 'pr_status',
       'neoplasm_histologic_grade', 'inferred_menopausal_state',
       'primary_tumor_laterality'],
      dtype='object')
y_train preview:
            time  event
0     125.900000      1
1      72.900000      0
2     110.466667      0
3     186.400000      1
4      35.200000      1
...          ...    ...
1517  104.400000      0
1518   82.633333      1
1519   61.900000      1
1520  307.633333      0
1521  192.200000      0

[1522 rows x 2 columns]


In [3]:
df_train_clean = pd.concat([X_train_clin, y_train], axis=1).dropna()
df_train_encoded = pd.get_dummies(df_train_clean, drop_first=True)

cph = CoxPHFitter(penalizer=0.01)
cph.fit(df_train_encoded, duration_col='time', event_col='event')
print("CoxPH Clinical Model fitted successfully!")

df_train_mol = pd.concat([X_train_mol, y_train], axis=1).dropna() if X_train_mol is not None else None
df_train_comb = pd.concat([X_train_comb, y_train], axis=1).dropna()
cph.print_summary()

CoxPH Clinical Model fitted successfully!


<lifelines.CoxPHFitter: fitted with 1086 total observations, 478 right-censored observations>
             duration col = 'time'
                event col = 'event'
                penalizer = 0.01
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 1086
number of events observed = 608
   partial log-likelihood = -3725.43
         time fit was run = 2026-10-18 12:01:15 UTC

---
                           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                  
age_at_diagnosis           0.03      1.03      0.00            0.03            0.04                1.03                1.04
neoplasm_histologic_grade  0.16      1.18      0.07            0.03            0.29                1.03                1.34
tumor_size                 0.01      1.01      0.00            0.01            0.02                1.01                1.02
tumor_stage                0.32      1.38      0.08            0.17            0.47                1.18                1.60
magea8                     0.00      1.00      0.04           -0.07            0.07                0.93                1.08
magea8_mut                 0.32      1.38      0.41           -0.48            1.12                0.62                3.07

                           cmp to    z      p  -log2(p)
covariate                                              
age_at_diagnosis             0.00 9.60 <0.005     70.02
neoplasm_histologic_grade    0.00 2.42   0.02      6.02
tumor_size                   0.00 4.65 <0.005     18.22
tumor_stage                  0.00 4.18 <0.005     15.08
magea8                       0.00 0.09   0.93      0.10
magea8_mut                   0.00 0.78   0.44      1.20
---
Concordance = 0.66
Partial AIC = 7462.85
log-likelihood ratio test = 202.81 on 6 df
-log2(p) of ll-ratio test = 133.94

In [4]:
df_train_mol = pd.concat([X_train_mol, y_train], axis=1).dropna() if X_train_mol is not None else None
print("Molecular training shape after cleaning:", df_train_mol.shape if df_train_mol is not None else "N/A")

df_train_comb = pd.concat([X_train_comb, y_train], axis=1).dropna()
print("Combined training shape after cleaning:", df_train_comb.shape)

Molecular training shape after cleaning: (1203, 676)
Combined training shape after cleaning: (881, 682)


In [5]:
df_train_mol_encoded = pd.get_dummies(df_train_mol, drop_first=True)

selector = VarianceThreshold(threshold=0.01)
X_train_mol_filtered = pd.DataFrame(
    selector.fit_transform(df_train_mol_encoded.drop(columns=['time', 'event'])),
    columns=df_train_mol_encoded.drop(columns=['time', 'event']).columns[selector.get_support()],
    index=df_train_mol_encoded.index
)

df_train_mol_clean = pd.concat([X_train_mol_filtered, y_train], axis=1).dropna()

cph_mol = CoxPHFitter(penalizer=1.0)
cph_mol.fit(df_train_mol_clean, duration_col='time', event_col='event')
print("Molecular Cox Model fitted successfully!")

df_train_comb_encoded = pd.get_dummies(df_train_comb, drop_first=True)
X_train_comb_features = df_train_comb_encoded.drop(columns=['time', 'event'])
selector_comb = VarianceThreshold(threshold=0.01)
X_train_comb_filtered = pd.DataFrame(
    selector_comb.fit_transform(X_train_comb_features),
    columns=X_train_comb_features.columns[selector_comb.get_support()],
    index=df_train_comb_encoded.index
)

df_train_comb_clean = pd.concat([X_train_comb_filtered, y_train], axis=1).dropna()

cph_comb = CoxPHFitter(penalizer=1.0)
cph_comb.fit(df_train_comb_clean, duration_col='time', event_col='event')
print("Combined Cox Model fitted successfully!")

Molecular Cox Model fitted successfully!
Combined Cox Model fitted successfully!


In [7]:
df_test_clin_encoded = pd.get_dummies(pd.concat([X_test_clin, y_test], axis=1), drop_first=True).dropna()
c_index_clin = cph.score(df_test_clin_encoded, scoring_method="concordance_index")

X_test_mol_features = pd.get_dummies(X_test_mol, drop_first=True) if X_test_mol is not None else None
X_test_mol_filtered = X_test_mol_features[X_train_mol_filtered.columns]
df_test_mol_clean = pd.concat([X_test_mol_filtered, y_test], axis=1).dropna()
c_index_mol = cph_mol.score(df_test_mol_clean, scoring_method="concordance_index")

X_test_comb_features = pd.get_dummies(test_features, drop_first=True)
X_test_comb_filtered = X_test_comb_features[X_train_comb_filtered.columns]
df_test_comb_clean = pd.concat([X_test_comb_filtered, y_test], axis=1).dropna()
c_index_comb = cph_comb.score(df_test_comb_clean, scoring_method="concordance_index")

print(f"\n--- Test Set Concordance Index (Cox Models) ---")
print(f"Clinical-Only Model: {c_index_clin:.4f}")
print(f"Molecular-Only Model: {c_index_mol:.4f}")
print(f"Combined Model: {c_index_comb:.4f}")


--- Test Set Concordance Index (Cox Models) ---
Clinical-Only Model: 0.6155
Molecular-Only Model: 0.6238
Combined Model: 0.6527


In [7]:
X_train_clin_encoded_xgb = df_train_encoded.drop(columns=['time', 'event'])
y_train_clean = y_train.loc[X_train_clin_encoded_xgb.index]
y_train_xgb = y_train_clean['time'] * np.where(y_train_clean['event'] == 1, 1, -1)

dtrain_clin = xgb.DMatrix(X_train_clin_encoded_xgb)
dtrain_clin.set_label(y_train_xgb)

params = {
    'objective': 'survival:cox',
    'eval_metric': 'cox-nloglik',
    'seed': 42
}

bst_clin = xgb.train(params, dtrain_clin, num_boost_round=100)
print("XGBoost Survival Model trained successfully on Clinical Features!")

X_train_mol_encoded_xgb = df_train_mol_clean.drop(columns=['time', 'event'])
y_train_mol_clean = y_train.loc[X_train_mol_encoded_xgb.index]
y_train_mol_xgb = y_train_mol_clean['time'] * np.where(y_train_mol_clean['event'] == 1, 1, -1)

dtrain_mol = xgb.DMatrix(X_train_mol_encoded_xgb)
dtrain_mol.set_label(y_train_mol_xgb)

bst_mol = xgb.train(params, dtrain_mol, num_boost_round=100)
print("XGBoost Survival Model trained successfully on Molecular Features!")

X_train_comb_encoded_xgb = df_train_comb_clean.drop(columns=['time', 'event'])
y_train_comb_clean = y_train.loc[X_train_comb_encoded_xgb.index]
y_train_comb_xgb = y_train_comb_clean['time'] * np.where(y_train_comb_clean['event'] == 1, 1, -1)

dtrain_comb = xgb.DMatrix(X_train_comb_encoded_xgb)
dtrain_comb.set_label(y_train_comb_xgb)

bst_comb = xgb.train(params, dtrain_comb, num_boost_round=100)
print("XGBoost Survival Model trained successfully on Combined Features!")

XGBoost Survival Model trained successfully on Clinical Features!
XGBoost Survival Model trained successfully on Molecular Features!
XGBoost Survival Model trained successfully on Combined Features!


In [15]:
X_test_clin_xgb_enc = pd.get_dummies(X_test_clin, drop_first=True)
df_test_clin_xgb = pd.concat([X_test_clin_xgb_enc, y_test], axis=1).dropna()
dtest_clin_clean = xgb.DMatrix(df_test_clin_xgb.drop(columns=['time', 'event']))
preds_clin = bst_clin.predict(dtest_clin_clean)
c_index_xgb_clin = concordance_index(df_test_clin_xgb['time'], -preds_clin, df_test_clin_xgb['event'])

X_test_mol_xgb_enc = pd.get_dummies(X_test_mol, drop_first=True) if X_test_mol is not None else None
X_test_mol_xgb_filtered = X_test_mol_xgb_enc[X_train_mol_filtered.columns]
df_test_mol_xgb = pd.concat([X_test_mol_xgb_filtered, y_test], axis=1).dropna()
dtest_mol_clean = xgb.DMatrix(df_test_mol_xgb.drop(columns=['time', 'event']))
preds_mol = bst_mol.predict(dtest_mol_clean)
c_index_xgb_mol = concordance_index(df_test_mol_xgb['time'], -preds_mol, df_test_mol_xgb['event'])

X_test_comb_xgb_enc = pd.get_dummies(test_features, drop_first=True)
X_test_comb_xgb_filtered = X_test_comb_xgb_enc[X_train_comb_filtered.columns]
df_test_comb_xgb = pd.concat([X_test_comb_xgb_filtered, y_test], axis=1).dropna()
dtest_comb_clean = xgb.DMatrix(df_test_comb_xgb.drop(columns=['time', 'event']))
preds_comb = bst_comb.predict(dtest_comb_clean)
c_index_xgb_comb = concordance_index(df_test_comb_xgb['time'], -preds_comb, df_test_comb_xgb['event'])

print(f"\n--- Final Test Set C-index Summary ---")
print(f"Cox Baseline - Clinical:  {c_index_clin:.4f} | XGBoost: {c_index_xgb_clin:.4f}")
print(f"Cox Baseline - Molecular: {c_index_mol:.4f} | XGBoost: {c_index_xgb_mol:.4f}")
print(f"Cox Baseline - Combined:  {c_index_comb:.4f} | XGBoost: {c_index_xgb_comb:.4f}")


--- Final Test Set C-index Summary ---
Cox Baseline - Clinical:  0.6155 | XGBoost: 0.6583
Cox Baseline - Molecular: 0.6238 | XGBoost: 0.5966
Cox Baseline - Combined:  0.6527 | XGBoost: 0.6720


In [21]:
# --- 6. Random Survival Forest (RSF) Training & Evaluation ---
def get_sksurv_y(y_df):
    return np.array(list(zip(y_df['event'].astype(bool), y_df['time'])), 
                    dtype=[('Status', '?'), ('Time', '<f8')])

y_train_sksurv = get_sksurv_y(y_train)
y_test_sksurv = get_sksurv_y(y_test)

# Clinical RSF
rsf_clin = RandomSurvivalForest(n_estimators=100, min_samples_split=10, min_samples_leaf=5, random_state=42)
# Fix: Using X_train_clin_encoded_xgb and y_train_clean indices correctly
rsf_clin.fit(X_train_clin_encoded_xgb, y_train_clean.loc[X_train_clin_encoded_xgb.index].pipe(get_sksurv_y))
X_test_clin_rsf = pd.get_dummies(X_test_clin, drop_first=True).reindex(columns=X_train_clin_encoded_xgb.columns, fill_value=0)
c_index_rsf_clin = rsf_clin.score(X_test_clin_rsf, y_test_sksurv)

# Molecular RSF
rsf_mol = RandomSurvivalForest(n_estimators=100, min_samples_split=10, min_samples_leaf=5, random_state=42)
rsf_mol.fit(X_train_mol_encoded_xgb, y_train.loc[X_train_mol_encoded_xgb.index].pipe(get_sksurv_y))
X_test_mol_rsf = X_test_mol_xgb_filtered.reindex(columns=X_train_mol_encoded_xgb.columns, fill_value=0)
c_index_rsf_mol = rsf_mol.score(X_test_mol_rsf, y_test_sksurv)

# Combined RSF
rsf_comb = RandomSurvivalForest(n_estimators=100, min_samples_split=10, min_samples_leaf=5, random_state=42)
rsf_comb.fit(X_train_comb_encoded_xgb, y_train.loc[X_train_comb_encoded_xgb.index].pipe(get_sksurv_y))
X_test_comb_rsf = X_test_comb_xgb_filtered.reindex(columns=X_train_comb_encoded_xgb.columns, fill_value=0)
c_index_rsf_comb = rsf_comb.score(X_test_comb_rsf, y_test_sksurv)

# --- 7. Final Comprehensive Summary ---
print("\n" + "="*70)
print("             FINAL TEST SET C-INDEX COMPARISON SUMMARY             ")
print("="*70)
print(f"Clinical  -> Cox: {c_index_clin:.4f} | XGBoost: {c_index_xgb_clin:.4f} | RSF: {c_index_rsf_clin:.4f}")
print(f"Molecular -> Cox: {c_index_mol:.4f} | XGBoost: {c_index_xgb_mol:.4f} | RSF: {c_index_rsf_mol:.4f}")
print(f"Combined  -> Cox: {c_index_comb:.4f} | XGBoost: {c_index_xgb_comb:.4f} | RSF: {c_index_rsf_comb:.4f}")
print("="*70)


             FINAL TEST SET C-INDEX COMPARISON SUMMARY             
Clinical  -> Cox: 0.6155 | XGBoost: 0.6583 | RSF: 0.6629
Molecular -> Cox: 0.6238 | XGBoost: 0.5966 | RSF: 0.6020
Combined  -> Cox: 0.6527 | XGBoost: 0.6720 | RSF: 0.6451
